In [2]:
import os

# ==========================================
# 0. ⚙️ GLOBAL CONFIGURATION
# ==========================================
# Change these values to control the entire script
TARGET_ROWS = 400           # Total rows to generate (e.g., 200 or 1300000)
NUM_CORES = 15              # Number of CPU cores to use
DUPLICATE_RATIO = 0.3       # 30% of data will be duplicates
OUTPUT_FILE = 'dedupe_churn_results.csv'
SETTINGS_FILE = 'dedupe_churn_settings.settings'
TRAINING_JSON = 'dedupe_churn_training.json'

# ⚠️ WINDOWS FIX: Must be set using the config above BEFORE importing dedupe
os.environ['LOKY_MAX_CPU_COUNT'] = str(NUM_CORES)

# ==========================================
# IMPORTS (Must be after os.environ setup)
# ==========================================
import random
import datetime
import csv
import time
import multiprocessing
import json
import logging 
import threading 
import itertools 
import sys       
import dedupe
import dedupe.variables

# ==========================================
# 1. Helper Utilities
# ==========================================
def random_date(start_year=1950, end_year=2005):
    """Generates a random date between two years."""
    start = datetime.date(start_year, 1, 1)
    end = datetime.date(end_year, 12, 31)
    return start + datetime.timedelta(days=random.randint(0, (end - start).days))

def corrupt_string(s):
    """Introduces noise (typos/deletions) into a string."""
    if not s or len(s) < 3: return s
    s_list = list(s)
    if random.random() > 0.5:
        idx = random.randint(0, len(s_list) - 2)
        s_list[idx], s_list[idx+1] = s_list[idx+1], s_list[idx]
    else:
        idx = random.randint(0, len(s_list) - 1)
        del s_list[idx]
    return "".join(s_list)

def get_random_bank_acct():
    """Generates a fake IBAN-like string."""
    return f"IE{random.randint(10,99)}BOFI{random.randint(900000, 999999)}"

def spinner_task(stop_event):
    """Runs a visual spinner in the console to show activity."""
    spinner = itertools.cycle(['-', '/', '|', '\\'])
    while not stop_event.is_set():
        sys.stdout.write(next(spinner))
        sys.stdout.flush()
        sys.stdout.write('\b')
        time.sleep(0.1)

# ==========================================
# 2. Main Data Generator
# ==========================================
def generate_huge_dataset(target_rows, duplicate_ratio):
    """
    Generates synthetic data matching the 'GI_AGG_DATA_CHURN' schema.
    """
    # --- Data Pools ---
    firsts = ["James", "John", "Robert", "Michael", "William", "David", "Richard", "Joseph", "Thomas", "Mary", "Patricia", "Jennifer", "Linda", "Elizabeth", "Barbara", "Susan", "Jessica", "Sarah", "Karen", "Lisa"]
    lasts = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson", "Martin"]
    
    companies = ["Aviva", "Tesco", "Dunnes", "Ryanair", "Kerry Group", "CRH", "Smurfit Kappa", "DCC", "Kingspan", "Glanbia", "Bank of Ireland", "AIB", "SuperValu", "Centra", "Spar", "Lidl", "Aldi", "Eir", "Vodafone", "Three"]
    suffixes = ["Ltd", "PLC", "Limited", "Holdings", "Group", "Ireland", "Services", "Solutions"]
    
    occupations = ["Teacher", "Engineer", "Nurse", "Doctor", "Accountant", "Manager", "Director", "Sales", "Admin", "IT Consultant", "Driver", "Builder", "Farmer", "Retiree", "Student", "Civil Servant", "Technician"]
    
    streets = ["Main St", "High St", "Church Rd", "Seaview", "Oak Park", "Griffith Ave", "O'Connell St", "Grafton St", "Henry St", "Dame St", "Patrick St", "Shop St", "Eyre Square", "Oliver Plunkett St"]
    cities = ["Dublin", "Cork", "Galway", "Limerick", "Waterford", "Drogheda", "Dundalk", "Swords", "Bray", "Navan"]

    data_d = {}
    ground_truth = {}
    counter = 0
    
    num_base_records = int(target_rows / (1 + duplicate_ratio))
    print(f"   [Helper] Generating {num_base_records} unique records...")

    # --- A. Generate Unique Base Records ---
    for i in range(num_base_records):
        counter += 1
        
        # 10% Chance of being a Corporate Customer (Company Ind = 'C')
        is_corporate = random.random() < 0.10
        
        if is_corporate:
            name = f"{random.choice(companies)} {random.choice(suffixes)}"
            gender = None
            dob = None
            occ = None 
        else:
            fn = random.choice(firsts)
            ln = random.choice(lasts)
            suffix = random.randint(1, 999) 
            name = f"{fn} {ln}{suffix}" 
            gender = random.choice(['M', 'F'])
            dob = random_date().strftime("%Y-%m-%d")
            occ = random.choice(occupations)

        # Shared Fields
        street_num = random.randint(1, 999)
        addr = f"{street_num} {random.choice(streets)}, {random.choice(cities)}"
        bank = get_random_bank_acct() if random.random() > 0.2 else None 

        record = {
            'name_only': name,
            'gender': gender,
            'address': addr,
            'dob': dob,
            'occupation': occ,
            'bank_acct_no': bank
        }
        
        data_d[counter] = record
        ground_truth[counter] = [counter]
        
        if i % 100000 == 0 and i > 0: print(f"       ...{i} base records created")

    # --- B. Generate Duplicates ---
    num_dupes = target_rows - num_base_records
    print(f"   [Helper] Generating {num_dupes} duplicates...")
    
    ids_to_dupe = random.choices(list(data_d.keys()), k=num_dupes)

    for i, original_id in enumerate(ids_to_dupe):
        counter += 1
        original = data_d[original_id]
        new_rec = original.copy()
        
        # Scenario 1: Typos
        if random.random() > 0.6: 
            new_rec['name_only'] = corrupt_string(new_rec['name_only'])
        
        # Scenario 2: Address Change
        if random.random() > 0.7 and new_rec['bank_acct_no']:
            new_rec['address'] = f"{random.randint(1,999)} New Address Rd, {random.choice(cities)}"
            
        # Scenario 3: Missing Data
        if random.random() > 0.8: new_rec['dob'] = None
        if random.random() > 0.8: new_rec['occupation'] = None
        if random.random() > 0.8: new_rec['gender'] = None

        data_d[counter] = new_rec
        ground_truth[original_id].append(counter)
        
        if i % 50000 == 0 and i > 0: print(f"       ...{i} duplicates created")

    # --- C. Generate Auto-Training Data ---
    print("   [Helper] Generating training pairs...")
    match_pairs = []
    distinct_pairs = []
    
    # Matches
    multi_grps = [g for g in ground_truth.values() if len(g) > 1]
    for _ in range(500): 
        if not multi_grps: break
        grp = random.choice(multi_grps)
        if len(grp) >= 2:
            a, b = random.sample(grp, 2)
            match_pairs.append((data_d[a], data_d[b]))

    # Distincts
    all_ids = list(data_d.keys())
    for _ in range(500):
        a, b = random.sample(all_ids, 2)
        distinct_pairs.append((data_d[a], data_d[b]))
            
    return data_d, {"match": match_pairs, "distinct": distinct_pairs}

# ==========================================
# 3. Main Execution Block
# ==========================================
if __name__ == '__main__':
    multiprocessing.freeze_support()
    
    # Enable Detailed Logging
    logger = logging.getLogger()
    logger.setLevel(logging.INFO)
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    logger.addHandler(ch)
    
    # Force Fresh Start
    if os.path.exists(TRAINING_JSON):
        print(f"🗑️ Deleting old training file: {TRAINING_JSON} (Fresh Start)")
        os.remove(TRAINING_JSON)

    fields = [
        dedupe.variables.String('name_only', has_missing=True),
        dedupe.variables.String('gender', has_missing=True),
        dedupe.variables.String('address', has_missing=True),
        dedupe.variables.String('dob', has_missing=True),
        dedupe.variables.String('occupation', has_missing=True),
        dedupe.variables.String('bank_acct_no', has_missing=True)
    ]
    
    print(f"⚡ Parallel Mode: {NUM_CORES} Cores")

    print(f"🌪️ Generating Data ({TARGET_ROWS} Rows)...")
    t_gen = time.time()
    
    # ⚙️ USING GLOBAL VARIABLES HERE
    data_d, training_data = generate_huge_dataset(TARGET_ROWS, DUPLICATE_RATIO) 
    
    print(f"✅ Generated in {time.time()-t_gen:.2f}s")
    
    print("🧠 Initializing Dedupe...")
    t0 = time.time()
    
    deduper = dedupe.Dedupe(fields, num_cores=NUM_CORES)
    
    if os.path.exists(TRAINING_JSON):
        print(f"   Reading labeled examples from {TRAINING_JSON}...")
        with open(TRAINING_JSON, 'r') as f:
            deduper.prepare_training(data_d, training_file=f)
    else:
        print("   Using auto-generated training data...")
        deduper.prepare_training(data_d)
        deduper.mark_pairs(training_data)

    print("🎓 Starting active labeling...")
    print("   [Instructions] y: Yes, n: No, u: Unsure, f: Finished")
    try:
        dedupe.console_label(deduper)
    except dedupe.predicates.NoIndexError:
        pass 

    print(f"💾 Saving manual training to {TRAINING_JSON}...")
    with open(TRAINING_JSON, 'w') as tf:
        deduper.write_training(tf)

    # --- 🌀 SPINNER IMPLEMENTATION START ---
    print("🎓 Training Model...", end=" ") 
    
    stop_spinner = threading.Event()
    spinner_thread = threading.Thread(target=spinner_task, args=(stop_spinner,))
    spinner_thread.start()
    
    try:
        deduper.train()
    finally:
        stop_spinner.set()
        spinner_thread.join()
    # --- 🌀 SPINNER IMPLEMENTATION END ---

    print(f"\n✅ Trained in {time.time()-t0:.2f}s")
    
    with open(SETTINGS_FILE, 'wb') as sf:
        deduper.write_settings(sf)

    print("🧩 Clustering...")
    t1 = time.time()
    clustered_dupes = deduper.partition(data_d, threshold=0.5)
    print(f"✅ Clustered in {time.time()-t1:.2f}s")
    
    print("💾 Saving CSV...")
    output_rows = []
    for cluster_id, (members, scores) in enumerate(clustered_dupes):
        for member_id, score in zip(members, scores):
            row = data_d[member_id]
            output_rows.append({
                'Cluster ID': cluster_id,
                'Score': round(score, 3),
                'Name': row['name_only'],
                'Gender': row['gender'],
                'DOB': row['dob'],
                'Address': row['address'],
                'Occupation': row['occupation'],
                'Bank Acct': row['bank_acct_no']
            })
            
    with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
        if output_rows:
            writer = csv.DictWriter(f, fieldnames=output_rows[0].keys())
            writer.writeheader()
            writer.writerows(output_rows)
            
    print(f"🎉 Done. Saved to {OUTPUT_FILE}")

🗑️ Deleting old training file: dedupe_churn_training.json (Fresh Start)
⚡ Parallel Mode: 15 Cores
🌪️ Generating Data (400 Rows)...
   [Helper] Generating 307 unique records...
   [Helper] Generating 93 duplicates...
   [Helper] Generating training pairs...
✅ Generated in 0.01s
🧠 Initializing Dedupe...
   Using auto-generated training data...


Final predicate set:
Final predicate set:
SimplePredicate: (fingerprint, address)
SimplePredicate: (fingerprint, address)
Final predicate set:
Final predicate set:
LevenshteinCanopyPredicate: (2, name_only)
LevenshteinCanopyPredicate: (2, name_only)


🎓 Starting active labeling...
   [Instructions] y: Yes, n: No, u: Unsure, f: Finished


name_only : SuperValu Services
gender : None
address : 894 Seaview, Limerick
dob : None
occupation : None
bank_acct_no : IE26BOFI963321

name_only : SuperValu PLC
gender : None
address : 475 Seaview, Cork
dob : None
occupation : None
bank_acct_no : None

500/10 positive, 500/10 negative
Do these records refer to the same thing?
(y)es / (n)o / (u)nsure / (f)inished


 y


...

 f


💾 Saving manual training to dedupe_churn_training.json...
🎓 Training Model... -

Finished labeling


Final predicate set:
Final predicate set:
TfidfNGramCanopyPredicate: (0.6, name_only)
TfidfNGramCanopyPredicate: (0.6, name_only)
(LevenshteinCanopyPredicate: (1, name_only), LevenshteinCanopyPredicate: (3, address))
(LevenshteinCanopyPredicate: (1, name_only), LevenshteinCanopyPredicate: (3, address))



✅ Trained in 82.38s
🧩 Clustering...
✅ Clustered in 6.34s
💾 Saving CSV...
🎉 Done. Saved to dedupe_churn_results.csv
